[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/02_types_of_machine_learning.ipynb)

# Bài 02 — Phân nhóm các thuật toán và thuật ngữ cơ bản

Trong Machine Learning, dựa trên tính chất của dữ liệu và **cách học**, có thể chia thành các nhóm thuật toán chính:
- Học có giám sát (Supervised learning)
- Học không giám sát (Unsupervised learning)
- Học bán giám sát (Semi-supervised learning)
- *(bổ sung)* Học tự giám sát (Self-supervised learning) — nền tảng của các mô hình ngôn ngữ lớn
- Học củng cố (Reinforcement learning)

<img src="images/machine-learning-map.png" style="width:75%;height:75%;">

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import make_classification_2d, make_regression_1d
from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Học có giám sát (Supervised Learning)

Máy tính được cung cấp dữ liệu **có gắn nhãn**: dữ liệu huấn luyện gồm các cặp `(data, label)` — $(\mathbf x^{(i)}, y^{(i)})$. Mục tiêu: học hàm $f$ sao cho $f(\mathbf x) \approx y$ trên **dữ liệu mới**.

<img src="images/sp-learning.png" style="width:65%;height:65%;">

Đầu ra có thể là:
- **giá trị liên tục** → **hồi quy (Regression)**: dự đoán giá nhà, cân nặng, doanh số...
- **nhãn rời rạc** → **phân loại (Classification)**: email spam hay không, ảnh chó hay mèo, chữ số 0–9...

*(bổ sung)* Một số dạng khác: **ranking** (xếp hạng kết quả tìm kiếm, gợi ý), **dự báo chuỗi thời gian** (chương 25), **structured prediction** (dịch máy, nhận diện thực thể).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
X, y = make_regression_1d(80, kind="linear", noise=1.5)
axes[0].scatter(X[:, 0], y, s=15)
axes[0].plot([-3, 3], [2 * -3 + 1, 2 * 3 + 1], "r", lw=2)
axes[0].set_title("Regression: y là số thực")
Xc, yc = make_classification_2d("blobs", n=200)
axes[1].scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=15)
axes[1].set_title("Classification: y là nhãn (2 lớp)")
plt.show()

## 2. Học không giám sát (Unsupervised Learning)

Dữ liệu **không** được gắn nhãn; thuật toán tự tìm **cấu trúc** trong dữ liệu. Trong thực tế, dữ liệu không nhãn nhiều hơn rất nhiều so với dữ liệu có nhãn (gán nhãn tốn công) — vì vậy học không giám sát luôn có giá trị đặc biệt.

<img src="images/usp-learning.png" style="width:65%;height:65%;">

Các bài toán chính:
- **Clustering (phân cụm):** chia dữ liệu thành các nhóm tương tự nhau — vd: phân khúc khách hàng (chương 20).
- **Anomaly detection (phát hiện bất thường):** học quy luật "bình thường", cảnh báo dữ liệu lệch khỏi quy luật — vd: một tài khoản giao dịch hàng tỷ đồng lúc nửa đêm (chương 22).
- *(bổ sung)* **Dimensionality reduction (giảm chiều):** nén dữ liệu nhiều chiều, trực quan hoá — PCA, t-SNE (chương 21).
- *(bổ sung)* **Density estimation:** ước lượng phân phối của dữ liệu — GMM (chương 23).

In [ ]:
from sklearn.cluster import KMeans

from ml_course.datasets import make_customer_segments

seg = make_customer_segments(n=500)
Xs = seg[["annual_income_musd", "spending_score"]].to_numpy()
labels = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(Xs)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(Xs[:, 0], Xs[:, 1], s=12, color="gray")
axes[0].set_title("Dữ liệu KHÔNG nhãn")
axes[1].scatter(Xs[:, 0], Xs[:, 1], c=labels, cmap="tab10", s=12)
axes[1].set_title("K-Means tự tìm ra 5 nhóm khách hàng")
for ax in axes:
    ax.set_xlabel("thu nhập")
    ax.set_ylabel("điểm chi tiêu")
plt.show()

## 3. Học bán giám sát (Semi-Supervised Learning)

Một **phần nhỏ** dữ liệu có nhãn, phần lớn không nhãn — kết hợp hai phương pháp trên. Ví dụ: kho ảnh khổng lồ nhưng chỉ vài trăm ảnh được gán nhãn. Ý tưởng: dùng cấu trúc của dữ liệu không nhãn (các cụm) để "lan truyền" nhãn.

<img src="images/semi-sp.png" style="width:35%;height:35%;">

In [ ]:
from sklearn.semi_supervised import LabelSpreading

Xm, ym = make_classification_2d("moons", n=400, noise=0.1)
y_partial = np.full_like(ym, -1)                  # -1 = không có nhãn
labeled = rng.choice(len(ym), 8, replace=False)   # chỉ 8 điểm có nhãn!
y_partial[labeled] = ym[labeled]
model = LabelSpreading(kernel="knn", n_neighbors=7).fit(Xm, y_partial)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(Xm[:, 0], Xm[:, 1], c="lightgray", s=10)
axes[0].scatter(Xm[labeled, 0], Xm[labeled, 1], c=ym[labeled], cmap="coolwarm", s=120, edgecolor="k")
axes[0].set_title("Chỉ 8 điểm có nhãn")
axes[1].scatter(Xm[:, 0], Xm[:, 1], c=model.transduction_, cmap="coolwarm", s=10)
axes[1].set_title(f"Lan truyền nhãn: accuracy = {np.mean(model.transduction_ == ym):.2f}")
plt.show()

## 4. Học tự giám sát (Self-Supervised Learning) *(bổ sung)*

Tự **tạo nhãn từ chính dữ liệu**: che một phần dữ liệu rồi bắt model dự đoán phần bị che. Ví dụ: đoán **từ tiếp theo** trong câu (GPT), đoán từ bị che (BERT), ghép các mảnh ảnh. Nhờ không cần người gán nhãn, model có thể học từ hàng nghìn tỷ từ trên internet → đây là nền tảng của các mô hình ngôn ngữ lớn (chương 31, 34).

## 5. Học củng cố (Reinforcement Learning)

Một **tác tử (agent)** tương tác với **môi trường**, thực hiện **hành động**, nhận **phần thưởng** hoặc **hình phạt**, và học **chiến lược** để tối đa hoá tổng phần thưởng về lâu dài. Không có "đáp án đúng" cho từng bước — chỉ có tín hiệu thưởng (thường đến muộn).

<img src="images/Reinforcement-Learning.png" style="width:55%">

Ứng dụng điển hình: AlphaGo (thắng nhà vô địch cờ vây thế giới), robot, điều khiển, tối ưu quảng cáo, huấn luyện LLM theo phản hồi của con người (RLHF). Chương 35.

## 6. Thuật ngữ cơ bản *(bổ sung)*

| Thuật ngữ | Ý nghĩa | Ví dụ (dự đoán giá nhà) |
|---|---|---|
| **Sample / instance** | một điểm dữ liệu (một hàng) | một căn nhà |
| **Feature** | đặc trưng đầu vào (một cột) | diện tích, số phòng, quận |
| **Target / label** | đại lượng cần dự đoán | giá nhà |
| **Dataset** | tập các sample | 1 500 căn nhà |
| **Model** | hàm $f(\mathbf x; \boldsymbol\theta)$ | $\hat y = \mathbf w^\top\mathbf x + b$ |
| **Parameters** | tham số **học từ dữ liệu** | $\mathbf w, b$ |
| **Hyperparameters** | tham số **người chọn trước** khi học | learning rate, độ sâu cây, $\lambda$ |
| **Loss function** | đo sai số trên **một** mẫu | $(y - \hat y)^2$ |
| **Objective / cost** | hàm cần tối ưu trên toàn bộ dữ liệu (có thể + regularization) | MSE + $\lambda\lVert\mathbf w\rVert^2$ |
| **Training** | tìm tham số tối ưu objective | gradient descent |
| **Inference / prediction** | dùng model đã học trên dữ liệu mới | dự đoán giá căn nhà vừa rao bán |
| **Generalization** | khả năng hoạt động tốt trên dữ liệu **chưa thấy** | — mục tiêu thật sự của ML |

> **Parameter vs hyperparameter:** nếu nó được thay đổi bởi `fit()` → parameter (`coef_`); nếu bạn truyền vào constructor → hyperparameter (`Ridge(alpha=...)`).

In [ ]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)            # alpha: HYPERPARAMETER — ta chọn
X, y = make_regression_1d(100, kind="linear")
model.fit(X, y)
print("Parameters học được:", model.coef_, model.intercept_)
print("Hyperparameters    :", model.get_params())

## Tổng kết

| Nhóm | Dữ liệu | Mục tiêu | Ví dụ |
|---|---|---|---|
| Supervised | $(\mathbf x, y)$ có nhãn | dự đoán $y$ | regression, classification |
| Unsupervised | chỉ $\mathbf x$ | tìm cấu trúc | clustering, anomaly, PCA |
| Semi-supervised | ít nhãn + nhiều không nhãn | dự đoán $y$ | lan truyền nhãn |
| Self-supervised | không nhãn, tự tạo nhãn | học biểu diễn | GPT, BERT |
| Reinforcement | tương tác + phần thưởng | học chiến lược | AlphaGo, robot |

Ngoài cách phân loại theo phương thức học, ta còn phân loại theo **chức năng**: regression, classification, clustering, dimensionality reduction, ...

## 🧠 Kiểm tra nhanh

1. Dự đoán số lượng đơn hàng ngày mai là regression hay classification?
2. Gom nhóm bài báo theo chủ đề khi không có nhãn chủ đề — thuộc nhóm nào?
3. `max_depth` của Decision Tree là parameter hay hyperparameter?
4. Vì sao self-supervised learning có thể dùng lượng dữ liệu lớn hơn supervised rất nhiều?

## Tài liệu tham khảo

[1] *Machine Learning? A Quick Guide* · [2] Vũ Hữu Tiệp — [Phân nhóm các thuật toán Machine Learning](https://machinelearningcoban.com/2016/12/27/categories/)

---
<sub>Bài gốc: *Các nhóm thuật toán trong Machine Learning* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 2 (MIT License). Đã bổ sung: self-supervised learning, ví dụ code cho từng nhóm, bảng thuật ngữ, parameter vs hyperparameter.</sub>